# P4 lab: embeddings and similarity

Lesson: [`concepts/P4_embeddings_similarity.md`](../concepts/P4_embeddings_similarity.md), Part H.1. Concept card: [02 P4](../02_RAG_MASTER_LEARNING_PLAN.md#p4).

**How this lab works (blank-file first).**
- Each task has a **task cell**, an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK cell after your code. If it fails, read the message and fix your code.
- This notebook has **no solution code**. The CHECK cells compare fingerprints, so the expected answers stay hidden until you have predicted.
- Where a task says **Predict first**, fill the prediction cell **before** you compute anything.
- **No API calls** (no Groq, no Cohere). Never print values from `.env`. All models load from your local cache.
- Kernel: the project **`.venv` (Python 3.12)**.

## Setup (given)
Run this cell. It names the input file, defines the fingerprint helper `_fp`, and gives the three toy vectors of lesson Part F.

In [ ]:
import hashlib, json, os
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # use the local model cache only
PDF = r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf"

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

Q = [1, 2, 2]                                   # the query vector (toy)
DOCS = {"d1": [2, 1, 2], "d2": [0, 3, 4], "d3": [2, 4, 4]}
QUESTION = "In what year was the Paris Agreement adopted?"   # 06 seed D-002, used in Tasks 7-8
print("ready")

## Task 1: the three measures by hand, then in code

Write four functions in **plain Python** (no NumPy, no library similarity function):
- `dot(a, b)`: the sum of `a[i] * b[i]`;
- `norm(a)`: the length of `a`, which is `sqrt(dot(a, a))`;
- `cosine(a, b)`: `dot(a, b) / (norm(a) * norm(b))`;
- `l2(a, b)`: the length of `a - b`.

Before you run anything, compute `dot(Q, d1)` and `cosine(Q, d1)` on paper.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p4_measure.py).

In [ ]:
# CHECK Task 1 (do not edit)
import math
assert dot([1, 0], [0, 1]) == 0 and dot([1, 2], [3, 4]) == 11, "dot is the sum of element-wise products"
assert math.isclose(norm([3, 4]), 5.0), "norm([3, 4]) should be 5"
assert math.isclose(cosine([1, 0], [5, 0]), 1.0) and math.isclose(cosine([1, 0], [0, 7]), 0.0), "cosine ignores length"
assert math.isclose(l2([0, 0], [3, 4]), 5.0), "l2 is the length of the difference"
toy = {k: [round(float(dot(Q, d)), 4), round(float(cosine(Q, d)), 4), round(float(l2(Q, d)), 4)] for k, d in DOCS.items()}
assert _fp(toy) == "23d247f9ba58e66f", f"some toy value is not right yet: {toy}"
for k, v in toy.items():
    print(f"{k}: dot={v[0]}  cos={v[1]}  L2={v[2]}")
print("Task 1 OK")

## Task 2: three rankings

**Predict first.** Which document does **raw L2** put first? Fill the cell below.

In [ ]:
# PREDICTION
my_prediction_l2_first = None   # "d1", "d2" or "d3", BEFORE you rank

Now build three lists of document names, best first:
- `rank_dot`: highest dot first;
- `rank_cos`: highest cosine first;
- `rank_l2`: **lowest** L2 first (a distance: smaller is closer).

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p4_measure.py).

In [ ]:
# CHECK Task 2 (do not edit)
assert my_prediction_l2_first is not None, "write your prediction first (it is part of the evidence)"
assert _fp({"dot": rank_dot, "cos": rank_cos, "l2": rank_l2}) == "e2162def2a314501", \
    "a ranking is not right yet. Is L2 sorted lowest first?"
print("Task 2 OK. You predicted", my_prediction_l2_first, "| raw L2 puts first:", rank_l2[0], "| cosine puts first:", rank_cos[0])

**Why do cosine and raw L2 disagree here?** One sentence.

*Template:* "They disagree because the vectors have different ____; cosine ignores ____ but L2 does not."

*Your answer:*

## Task 3: normalise, then compare again

Write `unit(a)`: `a` divided by `norm(a)`. Then build:
- `l2_unit`: a dict `{name: l2(unit(Q), unit(d))}` for the three documents;
- `rank_l2_unit`: the names sorted by `l2_unit`, lowest first.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p4_measure.py).

In [ ]:
# CHECK Task 3 (do not edit)
import math, random
assert math.isclose(norm(unit([3, 4])), 1.0), "unit(a) must have length 1"
for k, d in DOCS.items():   # the identity of lesson D.3, on your own functions
    assert math.isclose(l2(unit(Q), unit(d)) ** 2, 2 - 2 * cosine(Q, d), abs_tol=1e-12), f"L2^2 != 2 - 2cos for {k}"
rnd = random.Random(0)
for _ in range(200):
    a, b = [rnd.gauss(0, 1) for _ in range(8)], [rnd.gauss(0, 1) for _ in range(8)]
    assert math.isclose(dot(unit(a), unit(b)), cosine(a, b), abs_tol=1e-12), "dot of unit vectors should equal cosine"
vals = [round(float(l2_unit[k]), 4) for k in ("d1", "d2", "d3")]
assert _fp({"rank": rank_l2_unit, "l2": vals}) == "da3c754a37cd89d2", f"not right yet: {rank_l2_unit}, {vals}"
print("Task 3 OK. Ranking by L2 on unit vectors:", rank_l2_unit, "| cosine:", rank_cos)

**The proof in your own words.** Two lines, no notes.

*Template:* "For unit vectors, L2 squared = ____. So a larger cosine always means a ____ L2, and the two rankings are ____."

*Your answer:*

## Task 4 (CR-08): what if the query doubles?

**Predict first.** Use `Q2 = [2, 4, 4]` (the query times 2). Which of the three rankings (dot, cosine, raw L2) will change?

In [ ]:
# PREDICTION
Q2 = [2 * x for x in Q]
my_prediction_rankings_that_change = None   # e.g. ["l2"], BEFORE you compute

Build `rank_dot_2`, `rank_cos_2` and `rank_l2_2` for `Q2` (same rules as Task 2).

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p4_measure.py).

In [ ]:
# CHECK Task 4 (do not edit)
assert my_prediction_rankings_that_change is not None, "write your prediction first"
assert _fp({"dot": rank_dot_2, "cos": rank_cos_2, "l2": rank_l2_2}) == "3d8a0e4bda63e13c", "a ranking for Q2 is not right yet"
changed = [n for n, a, b in (("dot", rank_dot, rank_dot_2), ("cos", rank_cos, rank_cos_2), ("l2", rank_l2, rank_l2_2)) if a != b]
print("Task 4 OK. Rankings that changed:", changed, "| you predicted:", my_prediction_rankings_that_change)

## Task 5: does the real model already output unit vectors?

1. Load `model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")`.
2. `has_normalize`: `True` if one of the model's modules is a `Normalize` module (look at the modules; do not type `True`).
3. Load the PDF (PyMuPDF, pages joined with `"\n"`, as in P1) into `full`, and build `windows`: plain 1,000-character slices with overlap 200, `[full[i:i+1000] for i in range(0, len(full), 800)]`.
4. `E`: the embeddings of all windows (NumPy array). `norms_default`: `(min, max)` of their lengths.
5. `model_raw`: the same model **without** its Normalize module (hint: `SentenceTransformer(modules=[...])`). `E_raw`: its embeddings of the windows; `norms_raw`: `(min, max)` of their lengths.

**Predict first** (in the markdown cell): will the raw lengths be close to 1?

*My prediction (before running):* the raw lengths will be about ____ because ____.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p4_measure.py).

In [ ]:
# CHECK Task 5 (do not edit)
assert len(windows) == 91 and len(windows[1]) == 1000 and windows[1] == full[800:1800], "windows: 91 plain slices, size 1000, step 800"
assert E.shape == (91, 384) and E_raw.shape == (91, 384), "one 384-d vector per window"
got = {"normalize": bool(has_normalize), "default": [round(float(x), 4) for x in norms_default],
       "raw": [round(float(x), 2) for x in norms_raw]}
assert _fp(got) == "6be51c05e53fc6f0", f"not right yet: {got}"
print("Task 5 OK:", got)

## Task 6: a pair that means the same vs an unrelated pair

**Predict first.** Write your guesses for the two cosines below, between -1 and 1.

In [ ]:
# PREDICTION
PAIR_SIMILAR = ("greenhouse gas emissions", "carbon dioxide in the atmosphere")   # the book's Ch1 example
PAIR_UNRELATED = ("greenhouse gas emissions", "a recipe for chocolate cake")
my_prediction_cos_similar = None     # BEFORE you embed
my_prediction_cos_unrelated = None

Build `cos_similar` and `cos_unrelated` with `model` (Task 5).

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p4_measure.py).

In [ ]:
# CHECK Task 6 (do not edit)
assert None not in (my_prediction_cos_similar, my_prediction_cos_unrelated), "write both predictions first"
assert _fp([round(float(cos_similar), 4), round(float(cos_unrelated), 4)]) == "f68341df9b9ab388", \
    f"not right yet: {cos_similar:.4f}, {cos_unrelated:.4f}"
print(f"Task 6 OK. similar {cos_similar:.4f} (you said {my_prediction_cos_similar}), "
      f"unrelated {cos_unrelated:.4f} (you said {my_prediction_cos_unrelated})")

## Task 7: retrieval is one query against every chunk (MC-002)

Question (06 seed D-002): `QUESTION`, given in the setup cell.

Build:
1. `gold`: the index of the **one** window that contains `adopted in 2015` once runs of whitespace are replaced by one space.
2. `q_vec`: the question embedded **with the same model** as the windows.
3. `scores`: one cosine per window (shape `(91,)`).
4. `top5`: the indices of the 5 highest scores, best first. `gold_rank`: the 1-based rank of `gold`.

**Predict first** (in the markdown cell): what is the shape of `scores`?

*My prediction (before running):* `scores` has shape ____ because ____.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p4_measure.py).

In [ ]:
# CHECK Task 7a (do not edit)
assert getattr(scores, "shape", None) == (91,), "one score per window: a vector, not a matrix"
assert _fp({"gold": int(gold), "top5": [int(i) for i in top5], "rank": int(gold_rank)}) == "831a74674ad1601b", \
    f"not right yet: gold={gold}, top5={[int(i) for i in top5]}, rank={gold_rank}"
print("Task 7a OK. top5:", [int(i) for i in top5], "gold rank:", int(gold_rank))

**Now the wrong model.** Embed the same question with `SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L3-v2", device="cpu")`, a **different** model that also gives 384 numbers. Score it against the same `E` (normalise the query first). Build `top5_wrong` and `gold_rank_wrong`.

**Predict first:** will Python raise an error?

In [ ]:
# PREDICTION
my_prediction_error = None   # True or False, BEFORE you run

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p4_measure.py).

In [ ]:
# CHECK Task 7b (do not edit)
assert my_prediction_error is not None, "write your prediction first"
assert _fp({"top5": [int(i) for i in top5_wrong], "rank": int(gold_rank_wrong)}) == "d6d3ea759e232223", \
    f"not right yet: {[int(i) for i in top5_wrong]}, {gold_rank_wrong}. Did you normalise the query and use E from the L6 model?"
print("Task 7b OK. No error; gold window fell from rank", int(gold_rank), "to", int(gold_rank_wrong))

## Task 8: bi-encoder shortlist, cross-encoder rerank

1. `shortlist`: the 10 best windows by `scores` (Task 7), best first.
2. Load `ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")` (the reranker of `reranking.ipynb` cell 27).
3. Score each `(QUESTION, window)` pair of the shortlist. `n_pairs_scored`: how many pairs you scored.
4. `reranked_top3`: the 3 shortlist indices with the highest cross-encoder scores, best first.

**Predict first** (markdown cell): will the gold window move up?

*My prediction (before running):* the gold window will ____ because ____.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p4_measure.py).

In [ ]:
# CHECK Task 8 (do not edit)
assert list(shortlist[:5]) == list(top5), "the shortlist starts with your Task 7 top5"
assert _fp({"top3": [int(i) for i in reranked_top3], "pairs": int(n_pairs_scored)}) == "cf65c3c424d21f94", \
    f"not right yet: {[int(i) for i in reranked_top3]}, {n_pairs_scored}"
print("Task 8 OK. reranked top3:", [int(i) for i in reranked_top3], "| gold rank before:", int(gold_rank))

## Wrap-up: evidence for the log

Answer each in one line, no notes.

1. *Max sequence length, with its source:* "My model is ____; its max sequence length is ____; I read it from ____."
2. *The graph_rag inversion (`graph_rag.ipynb` cell 12):* "The code computes ____ from a score that is a ____, so the node explored first is the ____."
3. *Bi vs cross:* "A bi-encoder can store ____; a cross-encoder must ____ for every query."
4. *Asymmetry:* "For Cohere, chunks use `input_type` ____ and questions use ____."

> Tell the tutor when you are done. The tutor grades `coding_independence_0_3` (Tasks 1, 3, 7) and `debugging_0_3` (any failed CHECK you fixed yourself).